In [ ]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])



In [2]:
message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages
)

response.choices[0].message.content


"Hello and welcome.  It's great to meet you!  I'm GPT, a language model designed to assist and communicate with you. I'm here to help answer any questions you may have, provide information, or simply chat with you about any topic you're interested in. What's on your mind, feel like talking about something in particular or just getting to know each other?"

In [3]:
system_prompt = """
You are a funny humorous assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.
"""


In [4]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]


In [5]:
from bs4 import BeautifulSoup
import requests


# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]

In [6]:



def summarize(url):
    website = fetch_website_contents(url)
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages_for(website)
    )
    
    return response.choices[0].message.content


In [7]:
from IPython.display import Markdown, display

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))


In [8]:
display_summary("https://edwarddonner.com")


## Website Summary
This website belongs to Edward Donner, a code enthusiast, amateur music producer, and AI geek who also happens to be the CTO of Nebula.io. He's written some well-received Udemy courses and is probably going to spam you occasionally with good stuff if you give him your email.